<a href="https://colab.research.google.com/github/VGoma23/data-520-asean/blob/main/asean_gee_vessel_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

These imports need to be ran regardless of whether you're running the example code or the actual classification. It accesses the labeled data via google drive, assuming you have the following folders in your Google Drive directory:

```
\My Drive\Colab Notebooks\send_to_slack\
\My Drive\Colab Notebooks\mikey\
```

Those being Vibhu's 9-image test folder, and the 100-image sample that Vibhu sent to Mikey

In [44]:
from google.colab import drive
drive.mount('/content/drive')

medium_data_dir = '/content/drive/MyDrive/Colab Notebooks/ASEAN_pics/mikey'
small_data_dir = '/content/drive/MyDrive/Colab Notebooks/ASEAN_pics/sendtoslack'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# For personal file scraping
import glob
from pathlib import Path
import json


In [3]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='iuu-fishing-detections-asean')

In [4]:
pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 64.7 MB/s eta 0:00:00


In [5]:
# for coordinate translation
import rasterio
from rasterio.crs import CRS
from rasterio import warp

# Classification Example

This code was from [a video Neil sent](https://www.youtube.com/watch?v=qWaEfgWi21o) demonstrating pixel-level classification using the US national landcover dataset. I ran through it (up until the export part) and modified the testing point to be in SE Asia. Leaving it here for reference with the geemap / ee libraries.

In [ ]:
# import ee
# import geemap
# ee.Authenticate()
# ee.Initialize(project='iuu-fishing-detections-asean')

In [ ]:
# All this is coming from the video Neil sent;
# https://www.youtube.com/watch?v=qWaEfgWi21o

# ee.ImageCollection('ESA/WorldCover/v200').first()
# ee.ImageCollection("COPERNICUS/S2").filterDate('2023-01-01', '2023-01-31')

In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='iuu-fishing-detections-asean')

In [ ]:
Map = geemap.Map()
Map

In [ ]:
# old point that's somewhere in Cali,
# point = ee.Geometry.Point([-122.4439, 37.7538])
# point = ee.Geometry.Point([-87.7719, 41.8799]) # this one was alr commented

# image = (
#     ee.ImageCollection("LANDSAT/LC08/C01/T1_SR")
#     .filterBounds(point)
#     .filterDate("2016-01-01", "2016-12-31")
#     .sort("CLOUD_COVER")
#     .first()
#     .select("B[1-7]")
# )

# vis_params = {"min": 0, "max": 3000, "bands": ["B4", "B3", "B2"]}

# Map.centerObject(point, 8)
# Map.addLayer(image, vis_params, "Landsat")

# new point that will be somewhere on a coast in ASEAN, used for testing
# Lat, lon -> 12.648179, 100.988062
# point = ee.Geometry.Point([100.988062, 12.648179])

# training point that's somewhere in Cali
point = ee.Geometry.Point([-122.4439, 37.7538])

image = (
    ee.ImageCollection("COPERNICUS/S2")
    .filterBounds(point)
    # .filterDate("2016-01-01", "2016-12-31") # this code here selects the
    .filterDate("2016-01-01", "2016-12-31")
    .sort("CLOUDY_PIXEL_PERCENTAGE")
    .first()
    .select("B[1-7]")
)

vis_params = {"min": 0, "max": 3000, "bands": ["B4", "B3", "B2"]}

Map.centerObject(point, 8)
Map.addLayer(image, vis_params, "Sentinel-2")

In [ ]:
# ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo()
ee.Date(image.get("system:time_start")).getInfo()

In [ ]:
image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()

In [ ]:
nlcd = ee.Image("USGS/NLCD/NLCD2016").select("landcover").clip(image.geometry())
Map.addLayer(nlcd, {}, "NLCD")
Map

In [ ]:
# Make the training dataset.
points = nlcd.sample(
    **{
        "region": image.geometry(),
        "scale": 30,
        "numPixels": 5000,
        "seed": 0,
        "geometries": True,  # Set this to False to ignore geometries
    }
)

Map.addLayer(points, {}, "training", False)

In [ ]:
print(points.size().getInfo())

In [ ]:
print(points.first().getInfo())

In [ ]:
# Ideas to help make this more relevant to our project:
# # Use cloudcover layers in sentinel to mask out cloud-based false positives
# # Mess with which bands are used for training to see what gives better results
# # In the immediate moment, change the area to ASEAN waters

# # USE ee.geography.Point(coords, crs) TO TRANSLATE THE EPSG:____ COORDS INTO
# # ACTUAL COORDS

# # Also in the immediate moment, use the funny classifier to take US coast data
# # and predict landcover on ASEAN
# Done

# # Could try to do some classification that implements both the Sentinel and
# # Landsat data... actually idk how the time differences would work there.

# # Could try and implement austin's idea of averaging the GE images. With this
# # tutorial this shouldn't be that complicated in GEE... maybe

# # For realsies, this weekend I could try and make a file-scraper that creates
# # one aggregate file that contains all the boat locations, and gets all the
# # timestamps of the data in our files.

In [ ]:
# Use these bands for prediction.
bands = ["B1", "B2", "B3", "B4", "B5", "B6", "B7"]


# This property of the table stores the land cover labels.
label = "landcover"

# Overlay the points on the imagery to get training.
training = image.select(bands).sampleRegions(
    **{"collection": points, "properties": [label], "scale": 30}
)

# Train a CART classifier with default parameters.
trained = ee.Classifier.smileCart().train(training, label, bands)

In [ ]:
print(training.first().getInfo())

In [ ]:
# Classify the image with the same bands used for training.
result = image.select(bands).classify(trained)

# # Display the clusters with random colors.
Map.addLayer(result.randomVisualizer(), {}, "classified")
Map

In [ ]:
# new image for testing
asean_point = ee.Geometry.Point([100.988062, 12.648179])

asean_image = (
    ee.ImageCollection("COPERNICUS/S2")
    .filterBounds(asean_point)
    # .filterDate("2016-01-01", "2016-12-31") # this code here selects the
    .filterDate("2016-01-01", "2016-12-31")
    .sort("CLOUDY_PIXEL_PERCENTAGE")
    .first()
    .select("B[1-7]")
)

In [ ]:
asean_result = asean_image.select(bands).classify(trained)

# # Display the clusters with random colors.
Map.addLayer(asean_result.randomVisualizer(), {}, "asean_classified")
Map

In [ ]:
landcover = result.set("classification_class_values", class_values)
landcover = landcover.set("classification_class_palette", class_palette)

In [ ]:
Map.addLayer(landcover, {}, "Land cover")
Map

In [ ]:
print("Change layer opacity:")
cluster_layer = Map.layers[-1]
cluster_layer.interact(opacity=(0, 1, 0.1))

In [ ]:
Map.add_legend(builtin_legend="NLCD")
Map

In [ ]:
geemap.ee_export_image_to_drive(
    landcover, description="landcover", folder="classification_practice_export", scale=900
)

In [ ]:
class_palette = nlcd.get("landcover_class_palette").getInfo()
class_palette

In [ ]:
class_values = nlcd.get("landcover_class_values").getInfo()
class_values

# Skylight Dataset examples

These following sections show some of the leftover process I used in acquiring and using the skylight dataset in GEE.

In [ ]:
# Okay from here on out its all me.

# Step 1: pull up literally any image from the dataset.
  # scrape for metadata file at data's root
  # pull the time, location, and crs from the file
  # create a new data point with the satellite, at given point with crs
  # scrape for featurecollection at  /layers/label/data.geojson
  # copy and paste onto the map

## First Image example

In [ ]:
# metadatas = {[{"ImageName": img.name, "time": json.load( open(str(img / "metadata.json")))['time_range'],
#       "bounds" : str(img / "layers/label/data.geojson"),
#       "crs" : len(json.load( open(str(img / "layers/label/data.geojson")))['features'])
#       # , "View": f'<img src="{img}" width="200" height="200">' # View is optional
#      } for img in Path(f"{small_data_dir}").glob("*")]}

# first_path = Path(f"{small_data_dir}").glob("*")

first_metadata = json.load( open(str(small_data_dir + "/1186117_1894101_158904" + "/metadata.json")))
first_img_name = open(str(small_data_dir + "/1186117_1894101_158904" + "/image_name_from_siv.txt")).read()[:-5]
# 1186117_1894101_158904

first_metadata
first_img_name
# first_img_id = first_img_name[11:27] + first_img_name[45:] + "_" + first_img_name[38:44]
# first_img_id

In [ ]:
# src_crs = CRS.from_string(first_metadata['projection']['crs'][:])
# dst_crs = CRS.from_epsg(4326)

# x,y = warp.transform(src_crs, dst_crs, [first_lon], [first_lat])
# print(x, y)
# reprojected_point = ee.Geometry.Point([x[0],y[0]])


In [ ]:
first_rectangle_bounds = ee.Geometry.Rectangle(
    # [first_metadata['bounds'][0]*5, first_metadata['bounds'][3]*-5,
    #  first_metadata['bounds'][2]*5, first_metadata['bounds'][1]*-5],
    [first_metadata['bounds'][0], first_metadata['bounds'][1],
     first_metadata['bounds'][2], first_metadata['bounds'][3]],
    ee.Projection(first_metadata['projection']['crs'],
     [10, 0, 0, 0, -10, 0])
    , True, False
    )
first_rectangle_bounds.centroid()

In [ ]:
first_dateRangeStart = ee.Date(first_metadata['time_range'][0])
first_dateRangeEnd = ee.Date(first_metadata['time_range'][1])


# 1st image id: COPERNICUS/S2_HARMONIZED/20230603T154549_20230603T154551_T17QRU
# PRODUCT_ID is the name of the field
#                                        20230603T154549_20230603T191134_T17QRU
# first_image = ee.Image("COPERNICUS/S2_HARMONIZED/20230603T154549_20230603T154551_T17QRU")

first_image = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    # .filterBounds(first_point)
    # .filterDate("2016-01-01", "2016-12-31") # this code here selects the
    .filterDate(first_dateRangeStart, first_dateRangeEnd)
    # .sort("CLOUDY_PIXEL_PERCENTAGE")
    # .first()
    .filter(ee.Filter.stringContains('PRODUCT_ID',first_img_name)) # 20230603T154549_20230603T191134_T17QRU
    # bands with spatial of 10 may be easier for now; b2-4, 8
    .select("B[2-4]", "B8")
    .first()
    .clip(first_rectangle_bounds)
)

first_image

In [ ]:
# first_easting = (first_metadata['bounds'][2]+first_metadata['bounds'][0])*5
# first_northing = (first_metadata['bounds'][3]+first_metadata['bounds'][1])*-5
# first_point = ee.Geometry.Point([first_easting, first_northing]
#                        , first_metadata['projection']['crs']
#                        )

# first_easting = (first_metadata['bounds'][2]+first_metadata['bounds'][0])/2
# first_northing = (first_metadata['bounds'][3]+first_metadata['bounds'][1])/2
# first_point = ee.Geometry.Point(
#     [first_easting, first_northing]
#     , ee.Projection(first_metadata['projection']['crs'],
#      [10, 0, 0, 0, -10, 0])
#     )

first_point = first_rectangle_bounds.centroid()

first_point
# first_point.projection()


In [ ]:
Map = geemap.Map()

vis_params = {"min": 0, "max": 3000, "bands": ["B4", "B3", "B2"]}

Map.centerObject(first_point, 8)
Map.addLayer(first_image, vis_params, "Sentinel-2")
Map

## Collective image example

In [ ]:
# from here I'm trying to create an image collection of all the test data
# ...and it works!!!

In [ ]:
# image_collection = []
master_image = ee.ImageCollection([])

for img_folder in Path(f"{small_data_dir}").glob("*"):
  print(img_folder)
  metadata = json.load( open(str(img_folder / "metadata.json")))
  img_name = open(str(img_folder / "image_name_from_siv.txt")).read()[:-5]
  # would put lat/lon stuff here for labeled points... will get that soon
  # also here would create training non-vessel points for the unmarked pics
  # also also, need to clip image to bounds
  dateRangeStart = ee.Date(metadata['time_range'][0])
  dateRangeEnd = ee.Date(metadata['time_range'][1])

  rectangle_bounds = ee.Geometry.Rectangle(
    [metadata['bounds'][0], metadata['bounds'][1],
     metadata['bounds'][2], metadata['bounds'][3]],
    ee.Projection(metadata['projection']['crs'],
     [10, 0, 0, 0, -10, 0])
    , True, False
    )

  image = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    # .filterBounds(rectangle_bounds)
    .filterDate(dateRangeStart, dateRangeEnd)
    .filter(ee.Filter.stringContains('PRODUCT_ID',img_name))
    .select("B[2-4]", "B8")
    # TODO: need to add some extra checking here to make sure the above select
    #       doesn't return null. Specifically, 562176_... in the sample 7 is
    #       null even though a picture is in the dataset...
    # .first()
    # .clip(rectangle_bounds)
  )

  # image_collection.append(image)
  # if (image == Null):
  #   print("null img")
  Map.addLayer(image, vis_params, metadata['name'])
  master_image = master_image.merge(image)

# master_image = ee.ImageCollection(image_collection)
# Map.addLayer(master_image, vis_params, "combined images example")

master_image

In [ ]:
Map = geemap.Map()

# Map.addLayer(master_image, vis_params, "Sentinel-2 test images")
Map

## Collecting labels example

In [ ]:
withvessel_metadata = json.load( open(str(small_data_dir + "/561152_1314816_132052" + "/metadata.json")))
withvessel_img_name = open(str(small_data_dir + "/561152_1314816_132052" + "/image_name_from_siv.txt")).read()[:-5]
# 561152_1314816_132052
withvessel_features = json.load( open(str(small_data_dir + "/561152_1314816_132052" + "/layers/label/data.geojson")))['features']

withvessel_metadata
withvessel_features

In [ ]:
# src_crs = CRS.from_string(first_metadata['projection']['crs'][:])
# dst_crs = CRS.from_epsg(4326)

# x,y = warp.transform(src_crs, dst_crs, [first_lon], [first_lat])
# print(x, y)
# reprojected_point = ee.Geometry.Point([x[0],y[0]])


In [ ]:
withvessel_dateRangeStart = ee.Date(withvessel_metadata['time_range'][0])
withvessel_dateRangeEnd = ee.Date(withvessel_metadata['time_range'][1])

withvessel_rectangle_bounds = ee.Geometry.Rectangle(
    [withvessel_metadata['bounds'][0], withvessel_metadata['bounds'][1],
     withvessel_metadata['bounds'][2], withvessel_metadata['bounds'][3]],
    ee.Projection(withvessel_metadata['projection']['crs'],
     [10, 0, 0, 0, -10, 0])
    , True, False
    )

withvessel_image = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterDate(withvessel_dateRangeStart, withvessel_dateRangeEnd)
    .filter(ee.Filter.stringContains('PRODUCT_ID',withvessel_img_name))
    .select("B[2-4]", "B8")
    .first()
    .clip(withvessel_rectangle_bounds)
)

withvessel_image

In [ ]:
withvessel_point = withvessel_rectangle_bounds.centroid()

withvessel_point


In [ ]:
vessel_featurearray = []
# vessel_featurecollection = FeatureCollection

for feature in withvessel_features:
  vessel_featurearray.append(ee.Feature(ee.Geometry.Point(
      feature['geometry']['coordinates'],
      withvessel_rectangle_bounds.projection()
  )))

vessel_featurecollection = ee.FeatureCollection(vessel_featurearray)
vessel_featurecollection

In [ ]:
Map = geemap.Map()

vis_params = {"min": 0, "max": 3000, "bands": ["B4", "B3", "B2"]}

Map.centerObject(withvessel_point, 8)
Map.addLayer(withvessel_image, vis_params, "Image with vessels")
Map.addLayer(vessel_featurecollection)
Map

# Actual Classification

Things that can still be worked on:



1.   Look back at ee classification page to learn how to resize bands as necessary. [Link here](https://developers.google.com/earth-engine/guides/classification#colab-python:~:text=Python%20setup-,%23%20Define%20a%20function%20that%20scales%20and%20masks%20Landsat%208%20surface%20reflectance%20images.,-def%20prep_sr_l8(image)%3A%0A%20%20%22%22%22Scales), look at the "apply the scaling factors" section.
2.   ...



In [45]:
# can be small_data_dir (the send_to_slack sample)
# or medium_data_dir (the mikey/ 100-image folder)
training_folder = medium_data_dir

In [46]:
# bands = ['B8', 'B4', 'B3', 'B2']
bands = ['B12', 'B11', 'B8A', 'B8', 'B7', 'B6', 'B5', 'B4', 'B3', 'B2'] # 5 6 7 8A 11 12
vis_bands = ['B4', 'B3', 'B2']
label = 'vessel'
test_scale = 10

In [47]:
image_collection = []
# master_image = ee.ImageCollection([])

features_collection = ee.FeatureCollection([])

for img_folder in Path(f"{training_folder}").glob("*"):
  # print(img_folder)
  metadata = json.load( open(str(img_folder / "metadata.json")))
  img_name = open(str(img_folder / "image_name_from_siv.txt")).read()[:-5]
  dateRangeStart = ee.Date(metadata['time_range'][0])
  dateRangeEnd = ee.Date(metadata['time_range'][1])

  rectangle_bounds = ee.Geometry.Rectangle(
    [metadata['bounds'][0], metadata['bounds'][1],
     metadata['bounds'][2], metadata['bounds'][3]],
    ee.Projection(metadata['projection']['crs'],
     [10, 0, 0, 0, -10, 0])
    , True, False
    )

  image_c = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterDate(dateRangeStart, dateRangeEnd)
    .filter(ee.Filter.stringContains('PRODUCT_ID',img_name))
    .select(bands, bands)
    # .first()
    # .clip(rectangle_bounds)
  )

  if (image_c.size().getInfo() == 0):
    continue
  image = image_c.first().clip(rectangle_bounds)


  # would put lat/lon stuff here for labeled points... will get that soon
  features = json.load( open(str(img_folder / "layers/label/data.geojson")))['features']
  features_collection_unmapped = []
  for feature in features:
    features_collection_unmapped.append(ee.Feature(ee.Geometry.Point(
    feature['geometry']['coordinates'],
    rectangle_bounds.projection()),
      {label: 1}
    ))
  # features_collection.append(image.select(bands).sampleRegions(
  #   collection=features_collection_unmapped, properties=[label], scale=10
  # ))
  # also here would create training non-vessel points for the unmarked pics
  # TODO: also get non-vessel points from images with vessels in them
  if len(features) == 0:
    non_vessel_points_unlabeled = image.select(bands).sample(
      region= image.geometry(),
        scale = test_scale,
        numPixels = 20,
        # "seed": 0,
        geometries= False,  # Set this to False to ignore geometries
    )
  def set_label_0(feature):
    return feature.set(label, 0)
  # for feature in non_vessel_points_unlabeled:
  #   features_collection_unmapped.append(ee.Feature(ee.Geometry.Point(
  #   feature['geometry']['coordinates'],
  #   rectangle_bounds.projection()),
  #     {label: 0}
  #   ))
  non_vessel_points = non_vessel_points_unlabeled.map(set_label_0)
  # sample x amount of points, add them to features_collection with vessel = 0
  features_collection_unmapped_fc = ee.FeatureCollection(features_collection_unmapped)
  features_collection = features_collection.merge(image.select(bands).sampleRegions(
    collection=features_collection_unmapped_fc, properties=[label], scale=test_scale
  ))

  features_collection = features_collection.merge(non_vessel_points)

  image_collection.append(image)

  # Map.addLayer(image, vis_params, metadata['name'])

master_image = ee.ImageCollection(image_collection)
# master_image

# training_features = ee.FeatureCollection(features_collection)
# training_features
features_collection

In [37]:
Map = geemap.Map()

vis_params = {"min": 0, "max": 3000, "bands": vis_bands}

Map.addLayer(master_image, vis_params, "combined images example")

# Map.addLayer(training_features)

In [48]:
# Image 702464_1673216_74566 didnt work :/
# 707584_1673216_75212 didn't work, neither did 1043456_1602560_74906. Tbh idk
# how much of the total dataset is having this problem...

# if training on mikey split, should prob test 561152_1314816_132052 in smallDD

# this one is 563200_1319936_74897 in the mikey split. has 2 vessels.
test_rectangle_bounds = ee.Geometry.Rectangle(
    [33103, -613464,
     33682, -612886],
    ee.Projection("EPSG:32609", [10, 0, 0, 0, -10, 0]), True, False
    )

test_image = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterDate(ee.Date("2021-11-01T20:00:59+00:00"), ee.Date("2021-11-01T20:10:59+00:00"))
    .filter(ee.Filter.stringContains('PRODUCT_ID','S2B_MSIL1C_20211101T200559_N0301_R128_T09UUB_20211101T214915'))
    .select(bands, bands)
    .first()
    .clip(test_rectangle_bounds)
  )

test_image

# Map.addLayer(test_image, vis_params, "test image to be classified")

In [86]:
# Train a classifier with default parameters.
# the below classifier can be swapped out pretty easily below
# Note: currently testing smileRandomForest(40, bagFraction=0.35)
trained = ee.Classifier.smileRandomForest(2).train(features_collection, label, bands)

# Classify the image with the same bands used for training.
classified = test_image.classify(trained)
classified

In [87]:
Map.addLayer(test_image, vis_params, "test classified image")

Map.add_layer(
    classified,
    {'min': 0, 'max': 1, 'palette': ['blue', 'red']},
    'classification',
)

Map.centerObject(test_rectangle_bounds.centroid(), 13)

Map

Map(bottom=660524.0, center=[55.30409385956301, -131.61634014515408], controls=(WidgetControl(options=['positi…

In [ ]:
# ee.FeatureCollection('GOOGLE/EE/DEMOS/demo_landcover_labels')